This notebook can be [run on Google Colab using this link](https://colab.research.google.com/github/NEU-Silicon-Valley/CS7150-CA2/blob/main/CA2.4_ExtraCredit_MNIST.ipynb)

# Coding Assignment 2.4 (Extra Credit): Training a Neural Network on MNIST

This one is optional and worth up to **7 extra credit points**.

You'll train a small fully-connected network to recognise handwritten digits, then experiment with it. After that, make your own copy of the notebook that uses a **different dataset** and run the same experiments there. You submit both notebooks.

**Before you start:** if you opened this from the Colab link, use *File → Save a copy in Drive* so your work is kept. 

**Submitting:** write your answers in the answer cells at the end, and hand in a PDF of each notebook alongside the `.ipynb` file. The PDF is only there so we can read your answers quickly.

*Based on the TensorFlow Datasets [Keras MNIST example](https://www.tensorflow.org/datasets/keras_example), rewritten in PyTorch.*


In [ ]:
import time
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets

torch.manual_seed(7150)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using", device)


## Step 1: Load the data

MNIST is small enough to hold in memory, so we load it once as a tensor and scale the pixels from 0–255 down to 0–1. The training set is shuffled every epoch; the test set isn't.

For your second notebook, `FashionMNIST` and `KMNIST` drop straight into this cell in place of `MNIST`.


In [ ]:
train = datasets.MNIST("./data", train=True, download=True)
test = datasets.MNIST("./data", train=False, download=True)

x_train, y_train = train.data.float() / 255, train.targets
x_test, y_test = test.data.float() / 255, test.targets

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=128)
print(len(x_train), "training images,", len(x_test), "test images, each", tuple(x_train.shape[1:]))


## Step 2: Build and train the model

Two hidden layers of 32 and 16 units. The last layer outputs raw scores (logits) for the 10 digits; `CrossEntropyLoss` applies the softmax itself, so the model doesn't need one.

After every epoch we measure loss and accuracy on the test set and log them to TensorBoard under `run_name`. Give each experiment its own `run_name` so TensorBoard keeps them apart. The cell also reports how long training took, which you'll need for Q4.


In [ ]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 10),
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()


def evaluate(model, loader):
    """Average loss and accuracy over a whole dataset."""
    model.eval()
    total_loss = correct = n = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            total_loss += loss_fn(logits, y).item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            n += len(y)
    return total_loss / n, correct / n


run_name = "baseline_2hidden_lr1e-4"
writer = SummaryWriter(f"runs/{run_name}")

epochs = 40
history = {"loss": [], "test_loss": [], "test_acc": []}
start = time.perf_counter()
for epoch in range(epochs):
    model.train()
    running = seen = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item() * len(y)
        seen += len(y)
    test_loss, test_acc = evaluate(model, test_loader)
    history["loss"].append(running / seen)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)
    writer.add_scalar("loss/train", running / seen, epoch)
    writer.add_scalar("loss/test", test_loss, epoch)
    writer.add_scalar("accuracy/test", test_acc, epoch)
    print(f"epoch {epoch + 1:2d}/{epochs}  loss {running / seen:.4f}  test loss {test_loss:.4f}  test acc {test_acc:.4f}")
writer.close()
print(f"Training took {time.perf_counter() - start:.1f} s on {device}")


## Step 3: Look at the results


TensorBoard shows every run you've logged side by side, which makes the comparisons in Q2 and Q3 much easier. The plot below it is a static version of the same curves, which is what goes in your PDF.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs


In [ ]:
def plot_loss(history):
    plt.plot(history["loss"], label="train loss")
    plt.plot(history["test_loss"], label="test loss")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-entropy loss")
    plt.legend()
    plt.grid(True)
    plt.show()


plot_loss(history)
test_loss, test_acc = evaluate(model, test_loader)
print(f"Test loss {test_loss:.4f}, test accuracy {test_acc:.2%}")


A confusion matrix over the full test set shows which digits get mixed up. Rows are the true digit, columns the prediction.


In [ ]:
model.eval()
with torch.no_grad():
    preds = model(x_test.to(device)).argmax(1).cpu()
conf = torch.bincount(y_test * 10 + preds, minlength=100).reshape(10, 10)

plt.figure(figsize=(8, 7))
plt.imshow(conf, cmap="Blues")
for i in range(10):
    for j in range(10):
        plt.text(j, i, int(conf[i, j]), ha="center", va="center",
                 color="white" if conf[i, j] > conf.max() / 2 else "black")
plt.xticks(range(10))
plt.yticks(range(10))
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion matrix (full test set)")
plt.colorbar()
plt.show()


## Questions

**Q1 [1 point].** What is the MNIST dataset? Look into how it was collected, how big it is, and why it became so popular in machine learning.

> YOUR ANSWER:

**Q2 [2 points].** Train at least three different models and compare their accuracy, generalization, and speed. For example: one hidden layer, two hidden layers (the model above), and three hidden layers. You can also change the number of units in each layer. Include the learning curve for each model.

> YOUR ANSWER:

**Q3 [3 points].** Pick one model from Q2 and vary its hyperparameters. Try learning rates of 0.01, 0.001 and 0.0001 and describe what you see. Once you've settled on a good learning rate, try two different numbers of epochs and two different batch sizes.

> YOUR ANSWER:

**Q4 [1 point].** Run this notebook on the Explorer HPC as well as on Colab. The training cell prints how long training took; report both times along with the hardware each one ran on, and explain the difference.

> YOUR ANSWER:
